# Many Models Forecasting Demo

This notebook demonstrates how to conduct fine-grained model selection after running the `mmf.run_forecast` function. Before proceeding, ensure you have run the notebooks in [`/examples/monthly`](https://github.com/databricks-industry-solutions/many-model-forecasting/tree/main/examples/monthly). You can run this notebook on a serverless compute.

In [0]:
catalog = "mmf_demo_aym"  # Name of the catalog we use to manage our assets
db = "m4_advanced"             # Name of the schema we use to manage our assets (e.g. datasets)

In the `scoring_output` table, forecasts for each time series from every model are stored. Let's filter by a specific time series (e.g., `M1`) and examine the forecasts from all models.

In [0]:
scoring_output =  spark.sql(f"""
    SELECT model, unique_id, ds, y FROM {catalog}.{db}.monthly_scoring_output 
    WHERE unique_id='M1' ORDER BY model
    """)

display(scoring_output)

model,unique_id,ds,y
SKTimeProphet,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4974.514306169162, 4659.995909433403, 4460.624593342521)"
StatsForecastADIDA,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5096.531902036497, 5096.531902036497, 5096.531902036497)"
StatsForecastAutoArima,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4732.573163216812, 4654.769979177518, 4872.331808779956)"
StatsForecastAutoCES,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4841.742050170898, 4533.699951171875, 4561.9140625)"
StatsForecastAutoETS,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4996.233018106388, 4655.927940038661, 4690.571260237756)"
StatsForecastAutoMfles,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4659.570794222649, 4759.957476701542, 4851.841261865976)"
StatsForecastAutoTbats,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5327.045122005009, 5938.712694439512, 6181.269517137752)"
StatsForecastAutoTheta,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4801.26030136036, 4605.249231161135, 4665.008855892693)"
StatsForecastBaselineNaive,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(4010.0, 4010.0, 4010.0)"
StatsForecastBaselineSeasonalNaive,M1,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5120.0, 5960.0, 6560.0)"


This table contains forecasts from 44 different models, but we need to determine which one is best for making business decisions. This is where the `evaluation_output` table becomes useful. Let's filter by a specific time series (e.g., `M1`) and review the evaluation results (i.e., backtesting trials) from all models.

In [0]:
evaluation_output =  spark.sql(f"""
    SELECT model, unique_id, backtest_window_start_date, metric_name, metric_value, forecast, actual 
    FROM {catalog}.{db}.monthly_evaluation_output where unique_id='M1'
    AND model in ('ChronosBoltBase', 'TimesFM_2_5_200m', 'NeuralForecastAutoRNN', 'NeuralForecastAutoNHITS', 'StatsForecastBaselineNaive', 'StatsForecastAutoArima', 'SKTimeProphet')
    ORDER BY model, backtest_window_start_date
    """)

display(evaluation_output)

model,unique_id,backtest_window_start_date,metric_name,metric_value,forecast,actual
SKTimeProphet,M1,2022-01-31T00:00:00.000Z,smape,0.3209999384578449,"List(4789.336793953535, 4072.2877661859557, 3852.9680797753826)","List(5120.0, 5960.0, 6560.0)"
SKTimeProphet,M1,2022-02-28T00:00:00.000Z,smape,0.40546480270987,"List(4113.86057834463, 3882.373774278028, 3486.6949020148204)","List(5960.0, 6560.0, 4900.0)"
SKTimeProphet,M1,2022-03-31T00:00:00.000Z,smape,0.2591324529843621,"List(4100.030226402502, 3697.808825594929, 4359.218299745789)","List(6560.0, 4900.0, 4520.0)"
SKTimeProphet,M1,2022-04-30T00:00:00.000Z,smape,0.13250268407057178,"List(3764.498066840988, 4429.999293336433, 6566.612353384014)","List(4900.0, 4520.0, 7370.0)"
SKTimeProphet,M1,2022-05-31T00:00:00.000Z,smape,0.09227293103420119,"List(4678.976554021486, 6833.866039649894, 7656.945210122176)","List(4520.0, 7370.0, 9050.0)"
SKTimeProphet,M1,2022-06-30T00:00:00.000Z,smape,0.14643452389789804,"List(6781.596626936858, 7601.065172022074, 6481.417315813868)","List(7370.0, 9050.0, 7780.0)"
SKTimeProphet,M1,2022-07-31T00:00:00.000Z,smape,0.1376790531109447,"List(7697.2853190803835, 6583.268871503492, 4942.060295372989)","List(9050.0, 7780.0, 5380.0)"
SKTimeProphet,M1,2022-08-31T00:00:00.000Z,smape,0.10982767101690703,"List(6856.136023316074, 5230.838965379249, 5602.069413018608)","List(7780.0, 5380.0, 4700.0)"
SKTimeProphet,M1,2022-09-30T00:00:00.000Z,smape,0.08038614293103229,"List(5404.16357698319, 5786.0354682252055, 4624.612510682135)","List(5380.0, 4700.0, 4490.0)"
SKTimeProphet,M1,2022-10-31T00:00:00.000Z,smape,0.12796258591061282,"List(5798.694683833297, 4638.558030661061, 4623.117937936209)","List(4700.0, 4490.0, 4010.0)"


Based on the backtesting configuration, defined by the parameters `backtest_length`, `prediction_length`, and `stride` in the `mmf_sa.run_forecast` function, we obtain results from 10 backtesting trials for each model. For each trial, both forecasts and actual values are stored, enabling you to compute evaluation metrics based on residuals. Additionally, this table includes a built-in metric for quick assessment, which can be specified using the `metric` parameter. In this case, the metric is `smape`, and currently, `mae`, `mse`, `rmse`, `mape`, and `smape` are supported.

We compute the mean `smape` across 10 backtesting trials for each model and each time series. The model with the lowest mean `smape` is then selected for each time series, and its forecast is retrieved from the `forecast_output` table. Below is a SQL query that performs this selection.

In [0]:
forecast_best_model = spark.sql(f"""
    SELECT eval.unique_id, eval.model, eval.average_smape, score.ds, score.y
    FROM 
    (
      SELECT unique_id, model, average_smape,
      RANK() OVER (PARTITION BY unique_id ORDER BY average_smape ASC) AS rank
      FROM (
        SELECT unique_id, model, AVG(metric_value) AS average_smape
        FROM {catalog}.{db}.monthly_evaluation_output
        GROUP BY unique_id, model
        HAVING AVG(metric_value) IS NOT NULL
      )
    ) AS eval
    INNER JOIN {catalog}.{db}.monthly_scoring_output AS score 
      ON eval.unique_id=score.unique_id AND eval.model=score.model
    WHERE eval.rank=1
    ORDER BY eval.unique_id
    LIMIT 50
    """)

display(forecast_best_model)

unique_id,model,average_smape,ds,y
M1,StatsForecastBaselineSeasonalNaive,0.09188330565224587,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5120.0, 5960.0, 6560.0)"
M10,StatsForecastAutoArima,0.025042486924124908,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5407.310790357467, 5352.427969031163, 5294.062202784122)"
M11,StatsForecastAutoArima,0.003052778589836561,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(12743.0, 12999.999999999998, 13085.0)"
M12,SKTimeProphet,0.0031786676578988695,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2794.3075661905627, 2796.9654988698207, 2801.65263508063)"
M13,StatsForecastAutoArima,0.0024041327934189146,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2174.622186299542, 2172.344603218501, 2175.5579667670063)"
M14,StatsForecastAutoETS,0.0018315379326187685,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5012.698024307177, 5241.394759607733, 5266.1383238202)"
M15,StatsForecastBaselineSeasonalNaive,9.868075505686728E-4,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2657.2, 2657.8, 2667.9)"
M16,StatsForecastAutoArima,0.002361461252565674,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(14283.574315059503, 14493.577917046952, 14586.794956972386)"
M17,StatsForecastAutoArima,0.003847339833855086,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(7955.998740954994, 8138.376312134149, 8198.177664091716)"
M18,SKTimeProphet,0.0015257497024058716,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(6326.755924814871, 6343.50386381879, 6366.014665429789)"


These forecasts will be used to guide our business decisions. Let's count how many times each model was the best-performing one.

In [0]:
model_ranking = spark.sql(f"""
    SELECT model, count(*) as count
    FROM (
      SELECT unique_id, model, average_smape,
      RANK() OVER (PARTITION BY unique_id ORDER BY average_smape ASC) AS rank
      FROM (
        SELECT unique_id, model, AVG(metric_value) AS average_smape
        FROM {catalog}.{db}.monthly_evaluation_output
        GROUP BY unique_id, model
        HAVING AVG(metric_value) IS NOT NULL
      )
    ) WHERE rank=1 GROUP BY model 
    ORDER BY count DESC
    """)

display(model_ranking)

model,count
SKTimeProphet,19
StatsForecastAutoArima,17
StatsForecastAutoCES,13
StatsForecastAutoETS,10
StatsForecastBaselineSeasonalNaive,10
StatsForecastAutoTheta,8
StatsForecastAutoTbats,5
StatsForecastCrostonClassic,4
StatsForecastAutoMfles,3
StatsForecastCrostonSBA,3



On this dataset (M4 monthly), it appears that TimesFM models were the best performing models based on the number of time series they excelled in.

Exposing the `evaluation_output` and `forecast_output` tables in these formats provides great flexibility in model selection. For example, you can define your own evaluation metric to compare forecasting accuracy. You can aggregate metrics using a weighted average or even the median across backtesting trials. Additionally, you can retrieve forecasts from multiple models for each time series and ensemble them. All of these options simply require writing queries against these tables.

# Prediction Intervals Analysis

The MMF framework now produces `forecast_lower` and `forecast_upper` columns alongside point forecasts. These intervals come from:
- **Native distributional intervals** for StatsForecast models (ARIMA, ETS, Theta, etc.) and SKTime Prophet
- **Conformal prediction** as a fallback for foundation models (Chronos, TimesFM, Moirai) and any model without native intervals

Below we explode the array columns into individual time steps, compute coverage statistics, and visualize the intervals.

In [0]:
from pyspark.sql.functions import col, posexplode, arrays_zip, lit, avg, count, when, size

# Query evaluation output with interval columns
eval_intervals = spark.sql(f"""
    SELECT model, unique_id, backtest_window_start_date, metric_name, metric_value,
           forecast, actual, forecast_lower, forecast_upper
    FROM {catalog}.{db}.monthly_evaluation_output 
    WHERE unique_id = 'M1'
    AND model IN ('ChronosBoltBase', 'TimesFM_2_5_200m', 'StatsForecastAutoArima', 'SKTimeProphet')
    AND size(forecast_lower) > 0
    ORDER BY model, backtest_window_start_date
""")

# Explode arrays into individual time step rows
eval_exploded = (
    eval_intervals
    .select(
        "model", "unique_id", "backtest_window_start_date", "metric_name", "metric_value",
        posexplode(arrays_zip("forecast", "actual", "forecast_lower", "forecast_upper")).alias("step", "zipped")
    )
    .select(
        "model", "unique_id", "backtest_window_start_date", "metric_name", "metric_value", "step",
        col("zipped.forecast").alias("forecast"),
        col("zipped.actual").alias("actual"),
        col("zipped.forecast_lower").alias("forecast_lower"),
        col("zipped.forecast_upper").alias("forecast_upper"),
    )
)

display(eval_exploded)

model,unique_id,backtest_window_start_date,metric_name,metric_value,step,forecast,actual,forecast_lower,forecast_upper


In [0]:
# Compute interval coverage: % of actuals that fall within [lower, upper]
coverage_stats = (
    eval_exploded
    .filter(col("forecast_lower").isNotNull() & col("forecast_upper").isNotNull())
    .withColumn(
        "covered",
        when(
            (col("actual") >= col("forecast_lower")) & (col("actual") <= col("forecast_upper")),
            1
        ).otherwise(0)
    )
    .groupBy("model")
    .agg(
        avg("covered").alias("empirical_coverage"),
        count("*").alias("total_points"),
        avg(col("forecast_upper") - col("forecast_lower")).alias("avg_interval_width"),
    )
    .orderBy("model")
)

display(coverage_stats)

model,empirical_coverage,total_points,avg_interval_width


### How to Read Coverage Statistics

- **`empirical_coverage`**: The proportion of actual values that fell within the predicted interval. For a 95% interval, this should ideally be close to 0.95.
  - **Above 0.95** → intervals are conservative (wider than necessary, but safe)
  - **Below 0.95** → intervals are too narrow (under-confident)
- **`avg_interval_width`**: Narrower intervals with high coverage indicate a more precise model.
- **Native intervals** (StatsForecast, Prophet) typically achieve better calibration than conformal intervals because they model the error distribution directly.
- **Conformal intervals** (foundation models) are distribution-free and improve as more backtest windows are accumulated.

In [0]:
# Retrieve the best model's forecast WITH prediction intervals from scoring output
forecast_with_intervals = spark.sql(f"""
    SELECT eval.unique_id, eval.model, eval.average_smape,
           score.ds, score.y, score.forecast_lower, score.forecast_upper
    FROM 
    (
      SELECT unique_id, model, average_smape,
      RANK() OVER (PARTITION BY unique_id ORDER BY average_smape ASC) AS rank
      FROM (
        SELECT unique_id, model, AVG(metric_value) AS average_smape
        FROM {catalog}.{db}.monthly_evaluation_output
        GROUP BY unique_id, model
        HAVING AVG(metric_value) IS NOT NULL
      )
    ) AS eval
    INNER JOIN {catalog}.{db}.monthly_scoring_output AS score 
      ON eval.unique_id = score.unique_id AND eval.model = score.model
    WHERE eval.rank = 1
    ORDER BY eval.unique_id
    LIMIT 50
""")

display(forecast_with_intervals)

unique_id,model,average_smape,ds,y,forecast_lower,forecast_upper
M1,StatsForecastBaselineSeasonalNaive,0.09188330565224587,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5120.0, 5960.0, 6560.0)",List(),List()
M10,StatsForecastAutoArima,0.025042486924124908,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5407.310790357467, 5352.427969031163, 5294.062202784122)",List(),List()
M11,StatsForecastAutoArima,0.003052778589836561,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(12743.0, 12999.999999999998, 13085.0)",List(),List()
M12,SKTimeProphet,0.0031786676578988695,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2794.3075661905627, 2796.9654988698207, 2801.65263508063)",List(),List()
M13,StatsForecastAutoArima,0.0024041327934189146,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2174.622186299542, 2172.344603218501, 2175.5579667670063)",List(),List()
M14,StatsForecastAutoETS,0.0018315379326187685,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(5012.698024307177, 5241.394759607733, 5266.1383238202)",List(),List()
M15,StatsForecastBaselineSeasonalNaive,9.868075505686728E-4,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(2657.2, 2657.8, 2667.9)",List(),List()
M16,StatsForecastAutoArima,0.002361461252565674,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(14283.574315059503, 14493.577917046952, 14586.794956972386)",List(),List()
M17,StatsForecastAutoArima,0.003847339833855086,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(7955.998740954994, 8138.376312134149, 8198.177664091716)",List(),List()
M18,SKTimeProphet,0.0015257497024058716,"List(2023-01-31T00:00:00.000Z, 2023-02-28T00:00:00.000Z, 2023-03-31T00:00:00.000Z)","List(6326.755924814871, 6343.50386381879, 6366.014665429789)",List(),List()


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Pick a single time series for visualization
series_id = "M1"

# Get evaluation data for the best model on this series
best_model_row = (
    spark.sql(f"""
        SELECT model, AVG(metric_value) AS avg_metric
        FROM {catalog}.{db}.monthly_evaluation_output
        WHERE unique_id = '{series_id}'
        GROUP BY model
        HAVING AVG(metric_value) IS NOT NULL
        ORDER BY avg_metric ASC
        LIMIT 1
    """)
    .collect()
)

if best_model_row:
    best_model = best_model_row[0]["model"]
    print(f"Best model for {series_id}: {best_model}")

    # Get exploded backtest data for this model + series
    plot_df = (
        eval_exploded
        .filter((col("unique_id") == series_id) & (col("model") == best_model))
        .orderBy("backtest_window_start_date", "step")
        .toPandas()
    )

    if not plot_df.empty and plot_df["forecast_lower"].notna().any():
        fig, ax = plt.subplots(figsize=(14, 5))
        # Plot each backtest window
        for window, grp in plot_df.groupby("backtest_window_start_date"):
            x = range(len(grp))
            ax.plot(x, grp["actual"], "ko", markersize=3, label="Actual" if window == plot_df["backtest_window_start_date"].iloc[0] else "")
            ax.plot(x, grp["forecast"], "b-", linewidth=1.2, label="Forecast" if window == plot_df["backtest_window_start_date"].iloc[0] else "")
            ax.fill_between(
                x,
                grp["forecast_lower"].values,
                grp["forecast_upper"].values,
                alpha=0.2, color="blue",
                label="95% Interval" if window == plot_df["backtest_window_start_date"].iloc[0] else "",
            )
        ax.set_title(f"{series_id} — {best_model} (Backtest Windows with Prediction Intervals)")
        ax.set_xlabel("Forecast Horizon Step")
        ax.set_ylabel("Value")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No interval data available for visualization. Run the forecast pipeline with prediction_interval_level set.")
else:
    print(f"No evaluation data found for {series_id}")

Best model for M1: StatsForecastBaselineSeasonalNaive
No interval data available for visualization. Run the forecast pipeline with prediction_interval_level set.
